In [ ]:
import os
import cv2 as cv
import shutil as sh
import numpy as np
from google.colab import drive
from tensorflow.keras import layers, models
import tensorflow as tf
from tensorflow.keras.models import load_model
import gc
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
# filting and cleaning
drive.mount('/content/drive')
train_videos_path = "/content/drive/MyDrive/video_detection/Train_set"
videos_list_train= os.listdir(train_videos_path)
for i in range(len(videos_list_train)):
    videos_frames=os.path.join(train_videos_path,videos_list_train[i])
    frames=os.listdir(videos_frames)
    if len(frames)<100:
        print("removing")
        sh.rmtree(videos_frames)
    else:
        continue
print('*'*20)
test_videos_path= r"/content/drive/MyDrive/video_detection/Test_set"
videos_list_test= os.listdir(test_videos_path)
for i in range(len(videos_list_test)):
    videos_frames=os.path.join(test_videos_path,videos_list_test[i])
    frames=os.listdir(videos_frames)
    if len(frames)<100:
        print(videos_list_test[i])
        print("removing")
        sh.rmtree(videos_frames)
    else:
        continue
print('cleaned')
print(f'Trains videos:{len(videos_list_train)} clips')
print(f'Test videos:{len(videos_list_test)} clips')


In [ ]:
#preprocessing (resizing, normalizing, converting to grayscale) #
IMG_HEIGHT = 128
IMG_WIDTH = 128

def preprocess_clip_frames(clip_path):
    frame_files = sorted([f for f in os.listdir(clip_path) if f.endswith(".png")])

    frames = []

    for frame_file in frame_files:
        frame_path = os.path.join(clip_path, frame_file)

        img = cv.imread(frame_path)

        if img is None:
            continue

        img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        img = cv.resize(img, (IMG_WIDTH, IMG_HEIGHT))
        img = img.astype(np.float32) / 255.0

        frames.append(img)

    return np.array(frames, dtype=np.float32)
# motion map
def compute_motion_maps(frames_array):
    motion_maps = []

    for i in range(1, len(frames_array)):
        diff = np.abs(frames_array[i] - frames_array[i-1])
        motion_maps.append(diff)

    motion_maps = np.array(motion_maps, dtype=np.float32)
    motion_maps = np.expand_dims(motion_maps, axis=-1)

    return motion_maps


In [ ]:


all_preprocessed_clips = []
clip_names = sorted(os.listdir(train_videos_path))

for clip  in clip_names:
    clip_path = os.path.join(train_videos_path, clip)

    if os.path.isdir(clip_path):
        frames_array = preprocess_clip_frames(clip_path)
        all_preprocessed_clips.append(frames_array)

all_motion_map_clips = []

for i, frames_array in enumerate(all_preprocessed_clips):
    motion_maps = compute_motion_maps(frames_array)
    all_motion_map_clips.append(motion_maps)

print("*" * 20)
print("Number of processed clips:", len(all_preprocessed_clips))
print("Number of motion-map clips:", len(all_motion_map_clips))

In [ ]:
# creating sequences of motion maps for LSTM input
SEQUENCE_LENGTH = 10   # 9 motion maps as input, 1 motion map as target

def create_sequences_from_clip(motion_maps, sequence_length=10):
    X = []
    y = []

    for i in range(len(motion_maps) - sequence_length + 1):
        seq = motion_maps[i : i + sequence_length]

        X.append(seq[:sequence_length - 1])   # first 9 motion maps
        y.append(seq[sequence_length - 1])    # 10th motion map

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

all_X = []
all_y = []

for motion_maps in all_motion_map_clips:
    x_clip, y_clip = create_sequences_from_clip(motion_maps, SEQUENCE_LENGTH)
    all_X.append(x_clip)
    all_y.append(y_clip)

all_X = np.concatenate(all_X, axis=0)
all_y = np.concatenate(all_y, axis=0)

print("X shape:", all_X.shape)
print("y shape:", all_y.shape)

In [ ]:

X_train, X_val, y_train, y_val = train_test_split(
    all_X,
    all_y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

In [ ]:

del all_X
del all_y
del all_preprocessed_clips
del all_motion_map_clips

gc.collect()

In [ ]:


model = models.Sequential([
    layers.Input(shape=(9, 128, 128, 1)),

    layers.ConvLSTM2D(
        filters=16,
        kernel_size=(3, 3),
        padding="same",
        return_sequences=True,
        activation="tanh"
    ),
    layers.BatchNormalization(),

    layers.ConvLSTM2D(
        filters=8,
        kernel_size=(3, 3),
        padding="same",
        return_sequences=False,
        activation="tanh"
    ),
    layers.BatchNormalization(),

    layers.Conv2D(
        filters=1,
        kernel_size=(3, 3),
        padding="same",
        activation="sigmoid"
    )
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/video_detection/convlstm_motion_predictor.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=1,
    callbacks=callbacks
)

In [ ]:


best_model = load_model(
    "/content/drive/MyDrive/video_detection/convlstm_motion_predictor.keras"
)


predictions = best_model.predict(X_val, batch_size=4)

errors = np.mean(
    (predictions - y_val) ** 2,
    axis=(1, 2, 3)
)

print(errors.shape)

In [ ]:

plt.figure(figsize=(8, 5))
plt.hist(errors, bins=50)
plt.xlabel("Prediction Error (MSE)")
plt.ylabel("Number of Sequences")
plt.title("Validation Error Distribution")
plt.show()

print("Min error:", np.min(errors))
print("Max error:", np.max(errors))
print("Mean error:", np.mean(errors))
print("Std error:", np.std(errors))

In [ ]:

best_model = load_model("/content/drive/MyDrive/video_detection/convlstm_motion_predictor.keras")

val_pred = best_model.predict(X_val, batch_size=4, verbose=1)

val_errors = np.mean((val_pred - y_val) ** 2, axis=(1, 2, 3))

mean_error = np.mean(val_errors)
std_error = np.std(val_errors)

clip_threshold = mean_error + 3 * std_error
frame_threshold = mean_error + 2 * std_error

print("Min error:", np.min(val_errors))
print("Max error:", np.max(val_errors))
print("Mean error:", mean_error)
print("Std error:", std_error)

print("Clip Threshold:", clip_threshold)
print("Frame Threshold:", frame_threshold)

print("Clip-level normal:", np.sum(val_errors <= clip_threshold))
print("Clip-level above:", np.sum(val_errors > clip_threshold))

print("Frame-level normal:", np.sum(val_errors <= frame_threshold))
print("Frame-level above:", np.sum(val_errors > frame_threshold))

plt.figure(figsize=(8, 5))
plt.hist(val_errors, bins=50)
plt.axvline(clip_threshold, linestyle="--", label="Clip threshold")
plt.axvline(frame_threshold, linestyle="--", label="Frame threshold")
plt.xlabel("Prediction Error (MSE)")
plt.ylabel("Number of Sequences")
plt.title("Validation Error Distribution")
plt.legend()
plt.show()

In [ ]:

def preprocess_clip(clip_path, frame_size=(128, 128)):
    frames = []

    frame_files = sorted(os.listdir(clip_path))

    for frame_file in frame_files:
        frame_path = os.path.join(clip_path, frame_file)

        img = cv.imread(frame_path)

        if img is None:
            continue

        gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        gray = cv.resize(gray, frame_size)
        gray = gray.astype(np.float32) / 255.0

        frames.append(gray)

    return np.array(frames, dtype=np.float32)


def generate_motion_maps(frames):
    motion_maps = []

    for i in range(1, len(frames)):
        motion = np.abs(frames[i] - frames[i-1])
        motion_maps.append(motion[..., np.newaxis])

    return np.array(motion_maps, dtype=np.float32)

In [ ]:

results = []

clip_names = sorted(os.listdir(test_videos_path))

for clip_name in clip_names:
    clip_path = os.path.join(test_videos_path, clip_name)

    frames = preprocess_clip(clip_path)
    motion_maps = generate_motion_maps(frames)

    if len(motion_maps) < SEQUENCE_LENGTH:
        continue

    X_clip, y_clip = create_sequences_from_clip(motion_maps, SEQUENCE_LENGTH)

    pred = best_model.predict(X_clip, verbose=0, batch_size=4)

    clip_errors = np.mean((pred - y_clip) ** 2, axis=(1, 2, 3))

    mean_clip_error = float(np.mean(clip_errors))
    max_clip_error = float(np.max(clip_errors))

    prediction = "Anomaly" if max_clip_error > clip_threshold else "Normal"

    results.append({
        "Clip": clip_name,
        "Mean Error": mean_clip_error,
        "Max Error": max_clip_error,
        "Prediction": prediction
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Max Error", ascending=False).reset_index(drop=True)

display(results_df)

print(results_df["Prediction"].value_counts())

In [ ]:
def get_anomaly_segments(anomaly_frames, max_gap=2, min_length=3):
    """
    Groups nearby anomaly frames into segments.
    max_gap: allows small gaps between anomaly frames
    min_length: removes very short/noisy detections
    """
    if len(anomaly_frames) == 0:
        return []

    segments = []
    start = anomaly_frames[0]
    prev = anomaly_frames[0]

    for frame in anomaly_frames[1:]:
        if frame - prev <= max_gap:
            prev = frame
        else:
            if prev - start + 1 >= min_length:
                segments.append((start, prev))
            start = frame
            prev = frame

    if prev - start + 1 >= min_length:
        segments.append((start, prev))

    return segments

In [ ]:
CLIP_RANK = 23   # change this to inspect another test clip

clip_name = results_df.iloc[CLIP_RANK]["Clip"]
clip_path = os.path.join(test_videos_path, clip_name)

print("Clip:", clip_name)

frames = preprocess_clip(clip_path)
motion_maps = generate_motion_maps(frames)
X_clip, y_clip = create_sequences_from_clip(motion_maps, SEQUENCE_LENGTH)

pred = best_model.predict(X_clip, verbose=0, batch_size=4)

clip_errors = np.mean((pred - y_clip) ** 2, axis=(1, 2, 3))

frame_indices = np.arange(len(clip_errors)) + SEQUENCE_LENGTH

anomaly_mask = clip_errors > frame_threshold
anomaly_frames = frame_indices[anomaly_mask]

segments = get_anomaly_segments(
    anomaly_frames,
    max_gap=45,
    min_length=3
)

print("Total anomaly frames:", len(anomaly_frames))
print("Anomaly segments:", segments)

In [ ]:
#debugging section

plt.figure(figsize=(16, 5))

plt.plot(frame_indices, clip_errors, label="Frame error")
plt.axhline(clip_threshold, linestyle="--", label="Clip threshold")
plt.axhline(frame_threshold, linestyle="--", label="Frame threshold")

for start, end in segments:
    plt.axvspan(start, end, alpha=0.3)

plt.xlabel("Frame Index")
plt.ylabel("Prediction Error")
plt.title(f"Frame-by-frame anomaly timeline: {clip_name}")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:

frame_files = sorted(os.listdir(clip_path))

for seg_id, (start_frame, end_frame) in enumerate(segments):

    print(f"Segment {seg_id+1}: Frames {start_frame} to {end_frame}")

    segment_frames = list(range(start_frame, end_frame + 1))

    n_frames = len(segment_frames)
    cols = 5                              # 5 images per row
    rows = int(np.ceil(n_frames / cols))  # number of rows needed

    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, frame_idx in enumerate(segment_frames):

        img = cv.imread(os.path.join(clip_path, frame_files[frame_idx]))
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(f"Frame {frame_idx}")
        plt.axis("off")

    plt.suptitle(
        f"{clip_name} | Segment {seg_id+1} ({start_frame}-{end_frame})",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()